# Scrape Google Trends for Your Own Topic - Lane B (backup: the agent's output, captured)

**SISMID 2026 - Day 1, 3:30 session.** Yesterday's dengue pull, generalized: point the
same robust workflow at **your own** disease and place, and learn the **term vs topic**
distinction along the way.

> Each cell is a captured example of what a coding agent (Codex, Claude Code, or
> Antigravity CLI) produces from the **Lane A** prompts. Run it top to bottom as a
> backup, or read it as the target. The default topic is **flu in the US** (a real
> cached snapshot ships with it); edit two lines in Step 0 to make it yours.


## Step 0: the helper, and your topic

One robust fetch (retry-on-429, small stagger, cache fallback), a topic-resolver, and the
two lines you edit to choose your own query phrases and geography.


In [1]:
# --- Produced by the agent from the Plan A / Step 0 prompt ---
from pytrends.request import TrendReq   # pip install pytrends (in requirements.txt)
import pandas as pd, matplotlib.pyplot as plt, os, time, random

# ===== EDIT THESE TWO LINES for your own topic =====
MY_TERMS = ['influenza', 'flu', 'fever']   # a handful of query phrases, max 5
MY_GEO   = 'US'                            # e.g. 'US', 'MX', 'US-GA', 'BR'
# ===================================================
RECENT_TF = 'today 5-y'                    # past 5 years; last point is the current week

# a cached example (flu, US) so this notebook runs even if the live pull is blocked
CACHE_PATHS = [
    '../data/google_trends_flu_us_cached.csv',
    'data/google_trends_flu_us_cached.csv',
    './google_trends_flu_us_cached.csv',
]

def _norm(c):
    return c.strip().replace(' ', '_')

def gt_fetch(kw_list, timeframe, geo, tries=4):
    """Google Trends interest-over-time for a handful of terms/topics (max 5).
    kw_list entries can be literal phrases, additive queries like 'flu + gripe',
    or topic mids like '/m/0cycc'. Returns a tidy DataFrame (date + one column
    per entry), or None if Google keeps refusing. A first 429 is normal even from
    a Codespace (Azure IP); we wait and retry. The small random pause staggers the
    room so we do not all hit Google at once."""
    time.sleep(random.uniform(0, 3))   # stagger: do NOT count '3-2-1-everyone run'
    for attempt in range(tries):
        try:
            pt = TrendReq(hl='en-US', tz=360, retries=2, backoff_factor=0.5)
            pt.build_payload(kw_list, timeframe=timeframe, geo=geo)
            df = pt.interest_over_time()
            if df.empty:
                raise RuntimeError('empty frame (rate-limited)')
            df = df.drop(columns=[c for c in ['isPartial'] if c in df.columns]).reset_index()
            return df.rename(columns={c: _norm(c) for c in df.columns})
        except Exception as e:
            rate_limited = ('429' in str(e) or 'empty frame' in str(e)
                            or 'toomany' in type(e).__name__.lower())
            if rate_limited and attempt < tries - 1:
                wait = 12 * (attempt + 1)
                print(f'Rate-limited (attempt {attempt+1}/{tries}); waiting {wait}s and retrying...')
                time.sleep(wait)
                continue
            print(f'Live Google Trends pull failed ({type(e).__name__}): {e}')
            return None
    return None

def topic_mid(phrase):
    """Resolve a phrase to a Google Trends TOPIC entity (a Knowledge Graph 'mid'
    like '/m/0cycc') via pytrends suggestions. Returns (mid, title, type); falls
    back to the literal phrase if lookup fails."""
    try:
        s = TrendReq(hl='en-US', tz=360).suggestions(phrase)
        if s:
            return s[0]['mid'], s[0]['title'], s[0]['type']
    except Exception as e:
        print('suggestions lookup failed:', e)
    return phrase, phrase, 'raw term'

def load_cache():
    for p in CACHE_PATHS:
        if os.path.exists(p):
            print(f'Using cached example (flu, US): {p}')
            return pd.read_csv(p, parse_dates=['date'])
    raise FileNotFoundError('Cached example not found; check the data/ folder.')


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/vscode/.local/lib/python3.12/site-packages/traitlets/traitlets.py", line 651, in get
    value = obj._trait_values[self.name]
            ~~~~~~~~~~~~~~~~~^^^^^^^^^^^
KeyError: '_control_lock'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/vscode/.local/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/home/vscode/.local/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 347, in dispatch_control
    async with self._control_lock:
               ^^^^^^^^^^^^^^^^^^
  File "/home/vscode/.local/lib/python3.12/site-packages/traitlets/traitlets.py", line 706, in __get__
    return self.get(obj, cls)  # type:ignore[return-value]
           ^^^^^^^^^^^^^^^^^^
  File "/home/vscode/.local/lib/python3.12/site-packages/traitlets/traitlets.py", line 668,

## Step 1: pull your terms, live

A first 429 is normal; `gt_fetch` waits and retries. If it still gives up (e.g. the whole
room at once), we fall back to the cached flu/US example so the notebook still runs.


In [ ]:
# --- Produced by the agent from the Plan A / Step 1 prompt ---
series = gt_fetch(MY_TERMS, timeframe=RECENT_TF, geo=MY_GEO)
if series is None:
    series = load_cache()
cols = [c for c in series.columns if c != 'date']
print('rows:', len(series), '| last data point:', series['date'].max().date())
plt.figure(figsize=(10,4))
for c in cols:
    plt.plot(series['date'], series[c], label=c)
plt.legend(); plt.title(f'Google Trends terms ({MY_GEO})')
plt.ylabel('search interest (0-100)'); plt.xlabel('date'); plt.tight_layout(); plt.show()
series.tail()


## Step 2: term vs topic (and building a topic from phrases)

Google Trends gives you two different things:

- a **term** = the literal string. Misses synonyms, spellings, and other languages, and
  shows more zeros.
- a **topic** = Google's aggregated Knowledge Graph entity, which folds all of those in:
  more volume, fewer zeros.

Two ways to reach a topic-like signal, both just different `kw_list` entries to `gt_fetch`:

1. **combine phrases** with `" + "` (an additive / OR query) into one composite series;
2. **resolve the real topic entity** with `topic_mid()` and pull it by its `mid`.

*(This step is live-only and illustrative; the cached example above already shows the*
*individual terms.)*


In [ ]:
live = gt_fetch(MY_TERMS, RECENT_TF, MY_GEO)                     # individual terms
combo_query = ' + '.join(MY_TERMS)
combo = gt_fetch([combo_query], RECENT_TF, MY_GEO)               # phrases combined (OR)
mid, title, kind = topic_mid(MY_TERMS[0])
print(f"'{MY_TERMS[0]}' resolves to -> {title} ({kind}), mid={mid}")
topic = gt_fetch([mid], RECENT_TF, MY_GEO) if mid != MY_TERMS[0] else None  # real topic entity

if live is not None:
    tcols = [c for c in live.columns if c != 'date']
    pct_zero = (live[tcols] == 0).mean().mul(100).round(1)
    print('\n% zeros, individual terms:'); print(pct_zero)
    if combo is not None:
        ccol = [c for c in combo.columns if c != 'date'][0]
        print(f'% zeros, combined "+" query: {float((combo[ccol]==0).mean()*100):.1f}')
    if topic is not None:
        pcol = [c for c in topic.columns if c != 'date'][0]
        print(f'% zeros, topic entity {title}: {float((topic[pcol]==0).mean()*100):.1f}')
    plt.figure(figsize=(10,4))
    for c in tcols: plt.plot(live['date'], live[c], alpha=0.5, label=f'term: {c}')
    if combo is not None: plt.plot(combo['date'], combo[ccol], 'k', lw=2, label='combined "+"')
    if topic is not None: plt.plot(topic['date'], topic[pcol], 'r', lw=2, label=f'topic: {title}')
    plt.legend(fontsize=8); plt.title('Terms vs combined-phrases vs topic entity')
    plt.tight_layout(); plt.show()
else:
    print('Live pull blocked; skip the comparison (the cached terms above still work).')


## Step 3: the instability note

Google Trends is built from a sample of searches, so two pulls of the same query can
differ (a little on the public 0-100 endpoint, a lot on the raw Health Trends API). That
is the sampling issue from the afternoon session, and why retry and averaging matter.


In [ ]:
a = gt_fetch(MY_TERMS[:1], timeframe=RECENT_TF, geo=MY_GEO)
b = gt_fetch(MY_TERMS[:1], timeframe=RECENT_TF, geo=MY_GEO)
if a is not None and b is not None:
    m = a.merge(b, on='date', suffixes=('_1','_2'))
    col = [c for c in a.columns if c != 'date'][0]
    diff = (m[col+'_1'] - m[col+'_2']).abs()
    print('identical pulls? ', bool((diff == 0).all()), '| mean abs diff:', round(diff.mean(), 2))
else:
    print('Could not compare live (rate-limited). The point stands: repeat pulls can differ.')


## Step 4: sanity-check before you trust it


In [ ]:
# --- Produced by the agent from the Plan A / Step 4 prompt ---
print('geo         :', MY_GEO)
print('date range  :', series['date'].min().date(), 'to', series['date'].max().date())
print('n rows      :', len(series))
print('missing per column:'); print(series[cols].isna().sum())
print('which terms actually move (nonzero variance)?'); print(series[cols].std().round(2))


## Step 5: save your tidy CSV


In [ ]:
# --- Produced by the agent from the Plan A / Step 5 prompt ---
out = 'my_topic_search.csv'
series.to_csv(out, index=False)
print('saved', out, 'with', len(series), 'rows and columns', list(series.columns))


## Reflection

- You pulled a live signal for a topic \emph{you} chose, compared \textbf{terms vs a
  topic}, and checked it yourself.
- A combined `" + "` query, or the real topic entity, usually has fewer zeros and a
  cleaner seasonal shape than any single term. Prefer it when a term is sparse.
- Swap `MY_TERMS` and `MY_GEO` for a teammate's disease and place and rerun.

**If the whole room keeps getting blocked:** route through a proxy or VPN (see the slides
and `proxy-setup.md`); the cache is the floor, the proxy is how you still pull live.
